<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/04_integration_of_agentic_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import joblib

# Mount Google Drive
drive.mount('/content/drive')

# Verify your previously saved artifacts exist in Drive
base_path = '/content/drive/MyDrive/Malicious project work/'
expected_files = [
    'best_malware_model_lightgbm.pkl',
    'malware_scaler.pkl',
    'malware_label_encoder.pkl',
    'selected_feature_names.pkl'
]

print("Checking saved artifacts...")
for file in expected_files:
    full_path = os.path.join(base_path, file)
    if os.path.exists(full_path):
        print(f"  [FOUND] {file}")
    else:
        print(f"  [MISSING] {file} - Please check your path!")

In [ ]:
!pip install langchain langchain-openai langchain-community openai

In [ ]:
!pip install --upgrade google-colab

In [ ]:
import os
import joblib

# Load all artifacts using joblib
model = joblib.load(os.path.join(base_path, 'best_malware_model_lightgbm.pkl'))
scaler = joblib.load(os.path.join(base_path, 'malware_scaler.pkl'))
label_encoder = joblib.load(os.path.join(base_path, 'malware_label_encoder.pkl'))
selected_feature_names = joblib.load(os.path.join(base_path, 'selected_feature_names.pkl'))

print("All artifacts successfully loaded into memory using joblib!")

In [ ]:
import pandas as pd
import numpy as np

def predict_malware(features_dict):
    """
    Takes a dictionary of feature values, prepares a DataFrame matching selected features,
    scales it, and predicts whether the app is benign or malicious.
    """
    # Convert input to DataFrame
    input_df = pd.DataFrame([features_dict])

    # Ensure columns match selected features order (fill missing with 0)
    for col in selected_feature_names:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[selected_feature_names]

    # Scale features
    scaled_features = scaler.transform(input_df)

    # Predict
    pred_idx = model.predict(scaled_features)[0]
    pred_proba = model.predict_proba(scaled_features)[0]

    # Decode label
    predicted_label = label_encoder.inverse_transform([pred_idx])[0]
    confidence = float(pred_proba[pred_idx])

    return {
        "prediction": predicted_label,
        "confidence": confidence,
        "probabilities": {str(label_encoder.inverse_transform([i])[0]): float(p) for i, p in enumerate(pred_proba)}
    }

print("Prediction function created successfully!")

In [ ]:
!pip install langchain-classic langchain-google-genai -q

import os
import json
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

# 1. Set your Google AI Studio API key
os.environ["GEMINI_API_KEY"] = "...."

# 2. Wrap your predict_malware function as a tool
@tool
def analyze_mobile_app_malware(app_features_json: str) -> str:
    """
    Analyzes a mobile application's feature dictionary/string to detect whether it is malicious or benign
    using the LightGBM malware detection model. Returns prediction results, confidence score, and class probabilities.
    """
    try:
        if isinstance(app_features_json, str):
            features_dict = json.loads(app_features_json)
        else:
            features_dict = app_features_json

        result = predict_malware(features_dict)
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Error during malware classification: {str(e)}"

# 3. Initialize Gemini using the current gemini-3.8-flash model ID
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# 4. Define the security agent prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert cybersecurity and malware analysis assistant. Your role is to analyze mobile application characteristics, invoke the malware detection tool when needed, and provide clear, comprehensive security threat reports."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 5. Create the agent and executor
tools = [analyze_mobile_app_malware]
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Gemini Agent initialized successfully with gemini-3.8-flash!")

# 6. Run the Test
test_app_data = {
    "transact": 1,
    "onServiceConnected": 1,
    "bindService": 1,
    "attachInterface": 1,
    "SEND_SMS": 1,
    "READ_PHONE_STATE": 1
}

user_prompt = f"Please analyze this mobile application and let me know if it poses a threat: {json.dumps(test_app_data)}"
response = agent_executor.invoke({"input": user_prompt, "chat_history": []})

print("\n--- AGENT RESPONSE ---")
print(response["output"])

In [ ]:
test_app_data = {
    "transact": 1,
    "onServiceConnected": 1,
    "bindService": 1,
    "attachInterface": 1,
    "SEND_SMS": 1,
    "READ_PHONE_STATE": 1
}

user_prompt = f"Please analyze this mobile application and let me know if it poses a threat: {json.dumps(test_app_data)}"
response = agent_executor.invoke({"input": user_prompt, "chat_history": []})

print("\n--- AGENT RESPONSE ---")
print(response["output"])